# In-Class Activity: AI Workflows and AI Agents for a Construction Project
In this activity, we will use AI to take off quantities from foundation plans and to answer questions about a construction project from its specification, submittal log, and schedule, first as a fixed workflow, then as an AI agent that chooses its own tools, and finally with a skill and a HokieAI agent of your own.

In [ ]:
#@title ▶ Step 0 · Run me first (1 minute) { display-mode: "form" }
#@markdown Click ▶ and wait for the ✅ line. Paste your Gemini API key from MP5 into *api_key*, or leave it empty to use the Colab secret GEMINI_API_KEY.
#@markdown The default settings of every step run without a key (their outputs are stored); a key is needed only when you edit a prompt, a tool description, or the skill.
api_key = "" #@param {type:"string"}
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp8_agents", "aec_agent"
FOLDERS = ["mp8_agents"]                      # only this lab folder is downloaded, not the whole course repository

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("sparse-checkout", "set", *FOLDERS)
            and _git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", *FOLDERS], check=True)
try:
    import google.genai
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai"], capture_output=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_agent import lab
model = "gemini-3.5-flash-lite"
lab.setup(api_key=api_key, model=model)


## Part 1 · AI Workflows: Fixed Steps
In this part, we will explore an AI workflow: a task split into fixed steps, some done by an AI model and some by code. The task is a takeoff from five foundation plans (CMUs and grout), done once with one prompt and once as a workflow; both are scored against the ground truth.

In [ ]:
#@title ▶ Step 1a · Look at the Drawings { display-mode: "form" }
#@markdown Choose a drawing and run the cell to see it with its three takeoff questions. Some plans leave one side undimensioned: its length follows from the other dimensions.
drawing = "Foundation plan 1" #@param ["Foundation plan 1", "Foundation plan 2", "Foundation plan 3", "Foundation plan 4", "Foundation plan 5"]
lab.show_drawing(drawing)


In [ ]:
#@title ▶ Step 1b · One Prompt: The AI Model Does Everything { display-mode: "form" }
#@markdown Run the cell to send each drawing with its questions to the AI model (Gemini) in one prompt: the model reads the dimensions and computes every quantity.
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
prompt_text = """{problem}

End your reply with one final line per question, in this format:
{lines}"""
result_text = """**One prompt:** {right} of {n} quantities within tolerance of the ground truth (5 drawings)."""
table_text = """Ground truth vs. model output:"""
columns_text = """Drawing | Question | Quantity | Ground truth | Model output | Unit | Correct"""
lab.one_prompt(prompt_text=prompt_text, result_text=result_text, table_text=table_text, columns_text=columns_text)


In [ ]:
#@title ▶ Step 1c · Workflow: The AI Model Reads, Code Computes { display-mode: "form" }
#@markdown Run the cell to do the same takeoff as a workflow of two fixed steps:
#@markdown Step 1 (AI model): read the length of every side of the wall from the drawing (the prompt below).
#@markdown Step 2 (code): add the sides and apply the estimating conventions (for example, grouting at 32 in on center fills 8/32 of the cells).
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
read_prompt_text = """The attached foundation plan shows a CMU foundation wall; the dimensions are to the outside face of the wall. List the length of every side of the wall, going around the outside face. If a side is not dimensioned, work out its length from the other dimensions.
End your reply with one line, in this format:
SIDES: <length 1>, <length 2>, ... (in feet)"""
result_text = """**Workflow (AI model reads, code computes):** {right} of {n} quantities within tolerance of the ground truth (5 drawings)."""
steps_text = """Step 1 (AI model) and step 2 (code) for each drawing:"""
steps_columns_text = """Drawing | Sides read by the AI model (ft) | Total wall length (ft) | Ground truth (ft)"""
table_text = """Ground truth vs. workflow output:"""
columns_text = """Drawing | Question | Quantity | Ground truth | Workflow output | Unit | Correct"""
lab.workflow(read_prompt_text=read_prompt_text, result_text=result_text, steps_text=steps_text, steps_columns_text=steps_columns_text, table_text=table_text, columns_text=columns_text)


In [ ]:
#@title ▶ Step 1d · Compare One Prompt and the Workflow { display-mode: "form" }
#@markdown Run the cell after Steps 1b and 1c.
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
title_text = """Quantities within tolerance of the ground truth (15 quantities, 5 drawings)"""
columns_text = """Method | Correct | Model calls"""
empty_text = """Run Steps 1b and 1c first."""
lab.compare_workflow(title_text=title_text, columns_text=columns_text, empty_text=empty_text)


### 📝 Report question 1
> From Step 1d: Report the number of quantities within tolerance for one prompt and for the workflow, and include the comparison chart.

> From Steps 1b and 1c: Which quantities does one prompt get wrong, and why? Did the AI model read every side correctly in Step 1 of the workflow, including the undimensioned ones?

> Which steps of a takeoff would you give to an AI model and which to code (or a spreadsheet) on a real project, and why?

## Part 2 · AI Agents: Choose the Tools
In this part, we will explore an AI agent: an AI model that decides by itself which tool to call, with which inputs, and when it has enough information to answer. You choose which tools the agent may use, and every run answers the same six questions about the CEM4644 practice project (fictional): its specification, submittal log, and schedule are available only through the tools.

- **search_spec**: searches the specification (Section 03 30 00).
- **calculator**: evaluates an arithmetic expression exactly.
- **lookup_log**: reads rows of the submittal log or the schedule.

In [ ]:
#@title ▶ Step 2a · Look at the Project Documents { display-mode: "form" }
#@markdown Choose a document and run the cell to see what the tools read.
document = "specification (Section 03 30 00)" #@param ["specification (Section 03 30 00)", "submittal log", "schedule"]
lab.show_document(document)


In [ ]:
#@title ▶ Step 2b · Run the Agent with the Tools You Choose { display-mode: "form" }
#@markdown Tick the tools the agent may use and run the cell; every run is added to the table of runs. Run at least: no tools, each tool alone, and all three.
#@markdown Tool calling: the AI model does not run a tool itself; it asks for one (the tool's name and inputs), the notebook runs it, and the tool's output goes back to the model.
#@markdown UNKNOWN: the agent answered that it could not find the value; a wrong answer means it guessed.
#@markdown The system prompt and the tool descriptions below are what the agent reads. If you edit one, the run needs your Gemini key (Step 0).
search_spec = True #@param {type:"boolean"}
calculator = True #@param {type:"boolean"}
lookup_log = True #@param {type:"boolean"}
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
system_text = """You are an AI agent that answers questions about the CEM4644 practice project, a fictional building project. Project information (specification requirements, submittals, schedule) is available only through your tools. Do not guess a project value: if you cannot find it, answer UNKNOWN.
End your reply with the final answer on its own line, in this format: FINAL: <answer>
If the question asks for several quantities, give one FINAL line per quantity, for example: FINAL: Q1 = 832 CMU"""
search_spec_text = """Searches the project specification (Section 03 30 00, cast-in-place concrete) and returns the lines that best match the search words, for example the requirements for one element."""
calculator_text = """Evaluates an arithmetic expression exactly and returns the result. Supports + - * / ** ( ), ceil, floor, round, sqrt, min and max, for example: ceil(212 / 48)."""
lookup_log_text = """Returns the rows of a project log that contain the search text: log = submittals (the submittal log: status and the mix of each submittal) or schedule (the schedule: quantity and daily output of each activity)."""
result_text = """**Run {run} · tools: {tools}**
**Correct:** {right} of {n}   **UNKNOWN:** {unknown}   **Wrong:** {wrong}
**Tool calls:** {calls}   **Model calls:** {model_calls}"""
none_text = """no tools"""
table_text = """Ground truth vs. agent output (this run):"""
columns_text = """Question | Ground truth | Agent output | Result | Tools called"""
runs_text = """All runs so far:"""
runs_columns_text = """Run | Tools | Correct | UNKNOWN | Wrong | Tool calls | Model calls"""
lab.agent_tools(search_spec, calculator, lookup_log, system_text=system_text, search_spec_text=search_spec_text, calculator_text=calculator_text, lookup_log_text=lookup_log_text, result_text=result_text, none_text=none_text, table_text=table_text, columns_text=columns_text, runs_text=runs_text, runs_columns_text=runs_columns_text)


In [ ]:
#@title ▶ Step 2c · Inspect the Tool Calls { display-mode: "form" }
#@markdown Choose a question and run the cell to see, for every run of Step 2b, each tool the agent called, its inputs, and the tool's output.
question = "P5. Does the mix in submittal 03 30 00-02 meet this project's maximum w/cm for its element? Answer YES or NO." #@param ["P1. What is the maximum water-cementitious ratio (w/cm) for the foundation walls on this project?", "P2. A continuous footing is 187 ft long, 2 ft 7 in wide, and 1 ft 2 in deep. How many cubic yards of concrete are needed, including 7 % waste?", "P3. What is the review status of submittal 03 30 00-03?", "P4. How many working days are scheduled to place the slab-on-ground: the activity's quantity divided by its daily output, rounded up to a whole day?", "P5. Does the mix in submittal 03 30 00-02 meet this project's maximum w/cm for its element? Answer YES or NO.", "P6. Does the target slump of the mix in submittal 03 30 00-05 meet this project's maximum slump for its element? Answer YES or NO."]
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
run_text = """**Run {run} · tools: {tools}** · result: **{result}**"""
columns_text = """Step | Tool call | Tool output"""
answer_text = """**Agent output:** {answer}"""
empty_text = """No runs yet: run Step 2b first."""
lab.show_trace(question, run_text=run_text, columns_text=columns_text, answer_text=answer_text, empty_text=empty_text)


### 📝 Report question 2
> From Step 2b: Include the table of runs. For each tool, report which questions it was needed for.

> From Step 2c: For question P5, describe the agent's steps with all three tools. Without the tools it needed, did the agent answer UNKNOWN or guess?

> Which is more concerning on a real project, an agent that answers UNKNOWN or one that guesses, and how would you check an agent's answer?

## Part 3 · Skills: Instructions the Agent Loads When Needed
In this part, we will explore skills: written instructions, with a name and a description, that an agent loads only when a task needs them. The agent sees every skill's description; when the description matches the task, it loads the skill's full instructions (the body). The skill here holds the estimating conventions of Part 1, and the agent answers two takeoff questions (T1, T2) and one project question (P3).

In [ ]:
#@title ▶ Step 3a · Run the Agent with and without the Skill { display-mode: "form" }
#@markdown Run the cell once with *use_skill* ticked and once unticked, and compare the quantities correct and the questions the skill was loaded for.
#@markdown The agent has the calculator and the project logs; with *use_skill*, it also has the tool load_skill.
#@markdown Then make the description vague (for example, *Useful notes.*) and run again: does the agent still load the skill? If you edit the skill, the run needs your Gemini key (Step 0).
use_skill = True #@param {type:"boolean"}
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
system_text = """You are an AI agent that answers questions about the CEM4644 practice project, a fictional building project. Project information (specification requirements, submittals, schedule) is available only through your tools. Do not guess a project value: if you cannot find it, answer UNKNOWN.
End your reply with the final answer on its own line, in this format: FINAL: <answer>
If the question asks for several quantities, give one FINAL line per quantity, for example: FINAL: Q1 = 832 CMU"""
calculator_text = """Evaluates an arithmetic expression exactly and returns the result. Supports + - * / ** ( ), ceil, floor, round, sqrt, min and max, for example: ceil(212 / 48)."""
lookup_log_text = """Returns the rows of a project log that contain the search text: log = submittals (the submittal log: status and the mix of each submittal) or schedule (the schedule: quantity and daily output of each activity)."""
load_skill_text = """Loads the full instructions of one of the available skills by its name. Load a skill when its description matches the task."""
skills_list_text = """Available skills (load one with load_skill when its description matches the task):
- {name}: {description}"""
skill_name_text = """cmu-foundation-takeoff"""
skill_description_text = """Estimating conventions for CMU foundation walls: total wall length, number of CMUs, and grout volume. Use it for any CMU or grout quantity question."""
skill_body_text = """# CMU foundation takeoff: estimating conventions
- Total wall length = the sum of the outside dimensions of all sides (no deduction at the corners).
- Wall area = total wall length x wall height.
- One 8 in x 16 in (nominal) CMU covers 0.889 ft² of wall.
- CMUs to purchase = wall area / 0.889 ft², plus waste, rounded up to a whole CMU.
- Grout, all cells = CMUs before waste x grout per CMU, converted to cubic yards (27 ft³ = 1 CY), plus grout waste.
- Cells are 8 in apart, so grouting the cells at S in on center fills 8/S of the cells: grout = grout for all cells x 8 / S."""
result_text = """**Run {run} · skill: {skill}**
**Quantities correct:** {right} of {n}   **Skill loaded for:** {loaded}
**Tool calls:** {calls}   **Model calls:** {model_calls}"""
no_skill_text = """not available"""
none_loaded_text = """no question"""
table_text = """Ground truth vs. agent output (this run):"""
columns_text = """Question | Quantity | Ground truth | Agent output | Unit | Correct | Skill loaded"""
runs_text = """All runs so far:"""
runs_columns_text = """Run | Skill | Quantities correct | Skill loaded for | Tool calls | Model calls"""
lab.agent_skill(use_skill, system_text=system_text, calculator_text=calculator_text, lookup_log_text=lookup_log_text, load_skill_text=load_skill_text, skills_list_text=skills_list_text, skill_name_text=skill_name_text, skill_description_text=skill_description_text, skill_body_text=skill_body_text, result_text=result_text, no_skill_text=no_skill_text, none_loaded_text=none_loaded_text, table_text=table_text, columns_text=columns_text, runs_text=runs_text, runs_columns_text=runs_columns_text)


In [ ]:
#@title ▶ Step 3b · Create the Skill in HokieAI { display-mode: "form" }
#@markdown Create the skill of Step 3a in HokieAI, then send the takeoff questions T1 and T2 once without and once with the skill, and score both outputs.
skill = "with the skill" #@param ["without the skill", "with the skill"]
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
prompt_text = """{problems}

End your reply with one final line per question, in this format:
{lines}"""
steps_text = """1. In HokieAI ({url}), open **Account → My Skills → Create Skill**. Copy the skill's name, description, and body from Step 3a into the form, switch *Always On* off, and click *Create Skill*.
2. Choose *with the skill* or *without the skill* above and run this cell. Start a new chat; for *with the skill*, turn the skill on from the chat menu.
3. Copy the prompt below, paste it into the chat, and send.
4. Copy HokieAI's output with the **copy button** below its reply, paste it into the box below, and click *Score HokieAI output*."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**HokieAI, {skill}:** {right} of {n} quantities within tolerance of the ground truth."""
table_text = """Ground truth vs. HokieAI's output:"""
columns_text = """Question | Quantity | Ground truth | HokieAI's output | Unit | Correct"""
lab.hokieai_skill(skill, prompt_text=prompt_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text, columns_text=columns_text)


### 📝 Report question 3
> From Step 3a: Report the quantities correct with and without the skill, and the questions the skill was loaded for. Did the agent load the skill for P3? What happened with a vague description?

> From Step 3b: Report HokieAI's quantities correct without and with the skill.

> What estimating conventions or company rules from your own experience would you write as a skill, and what would its description say?

## Part 4 · Your Own HokieAI Agent
In this part, we will build an agent in HokieAI: instructions plus knowledge sources (the project files), asked the six questions of Part 2. Its output is scored against the ground truth and compared with the Gemini agent's runs.

In [ ]:
#@title ▶ Step 4a · Create Your HokieAI Agent { display-mode: "form" }
#@markdown Run the cell and follow the steps. The instructions below are a starting point: edit them in the box before pasting.
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
instructions_text = """You answer questions about the CEM4644 practice project, a fictional building project, using only the attached files: the specification excerpt (spec_03_30_00.txt), the submittal log (submittal_log.csv), and the schedule (schedule.csv). Show each value you look up and each calculation. Do not guess a project value: if the files do not contain it, answer UNKNOWN."""
create_steps_text = """1. Download the three project files with the buttons below.
2. In HokieAI ({url}), open **Explore Agents** and click *Create* next to *My Agents*; enter a name, for example *CEM4644 project agent*.
3. Under **Agent Instructions → Specific**, paste the instructions below.
4. Under **Knowledge Sources → Add → Upload Files**, upload the three files.
5. Click *Create*."""
lab.create_agent(instructions_text=instructions_text, create_steps_text=create_steps_text)


In [ ]:
#@title ▶ Step 4b · Ask Your HokieAI Agent { display-mode: "form" }
#@markdown Run the cell after Step 4a and follow the instructions. If you change your agent's instructions, start a new chat and score its output again.
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
prompt_text = """Answer the questions below. Reply with one line per question and nothing else, in this format:
P1 | <answer>

Questions:
{questions}"""
steps_text = """1. Open your agent in HokieAI ({url}) and start a new chat with it.
2. Copy the prompt below, paste it into the chat, and send.
3. Copy the agent's output with the **copy button** below its reply, and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth and with the Gemini agent's runs in Step 2b."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**Your HokieAI agent:** {right} of {n} correct, {unknown} UNKNOWN, {wrong} wrong."""
table_text = """Ground truth vs. your HokieAI agent's output:"""
columns_text = """Question | Ground truth | HokieAI agent output | Result"""
compare_text = """Your HokieAI agent vs. the Gemini agent's runs in Step 2b:"""
compare_columns_text = """Agent | Correct | UNKNOWN | Wrong"""
lab.hokieai_agent(prompt_text=prompt_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text, columns_text=columns_text, compare_text=compare_text, compare_columns_text=compare_columns_text)


### 📝 Report question 4
> From Step 4b: Report your agent's correct, UNKNOWN, and wrong answers, next to the Gemini agent's run with all three tools, and include the table.

> Which instructions did you change, and what changed in the output? For one wrong answer, explain from the output which value or calculation went wrong.

> What would you need to see before you let an agent answer project questions for your team?

### Data and model sources
- Practice project (specification excerpt, submittal log, schedule) and foundation plans: made for CEM4644 (fictional, not for construction); the specification excerpt follows the format of UFGS 03 30 00, U.S. Department of Defense, public domain, https://www.wbdg.org/dod/ufgs.
- AI model: Gemini (gemini-3.5-flash-lite), Google, via the Gemini API with your own key.
- Generative AI: GPT models via HokieAI (Virginia Tech).